In [1]:
import numpy as np
import pandas as pd

In [2]:
# ACS data
income = pd.read_csv('data/acs/ACS5Y2020/ACSDT5Y2020.B19001-Data.csv', skiprows=[1])
age = pd.read_csv('data/acs/ACS5Y2020/ACSDT5Y2020.B01001-Data.csv', skiprows=[0])

In [3]:
income['GEOID'] = income['GEO_ID'].apply(lambda x: x[9:])

# select only estimates columns and discard margin of errors
select_cols = [col for col in income.columns if col[-1]=="E"] + ['GEOID']
income = income[select_cols]

# remap income into coarse categories by combining columns
income['less_than_35k'] = income[[f'B19001_00{i}E' for i in range(2,8)]].sum(axis=1)
income['35k_75k'] = income[[f'B19001_00{i}E' for i in range(8,10)] + [f'B19001_0{i}E' for i in range(10,13)]].sum(axis=1)
income['75k_100k'] = income[['B19001_013E']].sum(axis=1)
income['greater_than_100k'] = income[[f'B19001_0{i}E' for i in range(14,18)]].sum(axis=1)
income['total_pop'] = income['B19001_001E']
income = income[['total_pop','GEOID','less_than_35k','35k_75k','75k_100k','greater_than_100k']]

# extract cbsa level marginal
income_cbsa = income.iloc[-1, 2:].astype(float)

# normalize into probabilities per each CBG
for col in income.columns[2:]:
    income[col] /= income['total_pop']

# drop the CBSA row and total_pop column
income = income.iloc[:-1, 1:]
income

,GEOID,less_than_35k,35k_75k,75k_100k,greater_than_100k
0,040130101021,0.085106,0.127660,0.182033,0.605201
1,040130101022,0.295756,0.087533,0.049072,0.567639
2,040130101023,0.149068,0.103520,0.064182,0.683230
3,040130101031,0.226510,0.275168,0.082215,0.416107
4,040130101032,0.118758,0.140351,0.170040,0.570850
...,...,...,...,...,...
3028,040219413001,0.680303,0.157576,0.107576,0.054545
3029,040219414011,0.319481,0.270130,0.044156,0.366234
3030,040219414012,0.585915,0.273239,0.056338,0.084507
3031,040219414013,0.243065,0.459709,0.228534,0.068692


In [4]:
income_cbsa

less_than_35k        416253.0
35k_75k              547471.0
75k_100k             237884.0
greater_than_100k    543611.0
Name: 3033, dtype: float64

In [5]:
age['GEOID'] = age['Geography']
cols_to_keep = [col for col in age.columns if 'Margin of Error' not in col]
age = age[cols_to_keep]


""" Remap age groups into coarse categories
The data is sex by age and there are many joint age categories, 
(e.g. "Estimate!!Total:!!Male:!!Under 5 years").
Since there are be many columns to combine, it's easier to pivot to long form
then split the column names by the delimiter, drop the sex info and remap by age group names
then finally do a group-by to sum up the estimates
"""
age = age.melt(id_vars='GEOID', value_vars=age.columns[3:-1])
age = age.query('variable != "Estimate!!Total:!!Male:" and variable != "Estimate!!Total:!!Female:"')
age['age'] = (
    age['variable']
      .str.rsplit('!!', n=1)
      .str[-1]
      .str.replace(r':$', '', regex=True)  # drop a trailing colon
      .str.strip()
)

# remap age groups
age_group_map = {
    # Under 18
    'Under 5 years': 'Under 18',
    '5 to 9 years': 'Under 18',
    '10 to 14 years': 'Under 18',
    '15 to 17 years': 'Under 18',

    # 18 to 24
    '18 and 19 years': '18 to 24',
    '20 years': '18 to 24',
    '21 years': '18 to 24',
    '22 to 24 years': '18 to 24',

    # 25 to 44
    '25 to 29 years': '25 to 44',
    '30 to 34 years': '25 to 44',
    '35 to 39 years': '25 to 44',
    '40 to 44 years': '25 to 44',

    # 45 to 66
    '45 to 49 years': '45 to 66',
    '50 to 54 years': '45 to 66',
    '55 to 59 years': '45 to 66',
    '60 and 61 years': '45 to 66',
    '62 to 64 years': '45 to 66',
    '65 and 66 years': '45 to 66',

    # 67 and above
    '67 to 69 years': '67 and above',
    '70 to 74 years': '67 and above',
    '75 to 79 years': '67 and above',
    '80 to 84 years': '67 and above',
    '85 years and over': '67 and above'
}

age['age_group'] = age['age'].map(age_group_map)

# group by cbg and age_group and sum all values
age = (age
       .groupby(['GEOID','age_group'], as_index=False)
       .agg(estimate=('value','sum'))
       .query('age_group != "Under 18"')) # drop Under 18 category since cuebiq >= 18

# pivot back into wide form
age = age.pivot(columns='age_group', index='GEOID', values='estimate')

# extract CBSA marginals (last row) and drop from cbg df
age_cbsa = age.iloc[-1]
age = age.iloc[:-1]

# normalize to get probability distributions per CBG
age = age[['18 to 24', '25 to 44', '45 to 66', '67 and above']].div(age.sum(axis=1), axis=0)
age.columns.name = None
age = age.reset_index()
age['GEOID'] = age['GEOID'].apply(lambda x: x[9:])

age

,GEOID,18 to 24,25 to 44,45 to 66,67 and above
0,040130101021,0.003542,0.319953,0.279811,0.396694
1,040130101022,0.006592,0.071852,0.555043,0.366513
2,040130101023,0.023019,0.000000,0.464151,0.512830
3,040130101031,0.045374,0.129004,0.556940,0.268683
4,040130101032,0.026382,0.236809,0.499372,0.237437
...,...,...,...,...,...
3028,040219413001,0.256350,0.389250,0.264028,0.090372
3029,040219414011,0.186841,0.299849,0.331994,0.181316
3030,040219414012,0.199516,0.408706,0.274486,0.117291
3031,040219414013,0.192424,0.283333,0.432323,0.091919


In [6]:
age_cbsa

age_group
18 to 24         444363.0
25 to 44        1337601.0
45 to 66        1262610.0
67 and above     663574.0
Name: 310M600US38060, dtype: float64

In [7]:
age_cbsa.name = 'pop'
income_cbsa.name = 'pop'
income_cbsa.index.name = 'hh_income'

age_cbsa.to_csv('data/processed/age_cbsa_margins_38060.csv')
income_cbsa.to_csv('data/processed/income_cbsa_margins_38060.csv')

In [8]:
# normalize
income_cbsa_distr = income_cbsa /  income_cbsa.sum()
age_cbsa_distr = age_cbsa /  age_cbsa.sum()

In [9]:
income_cbsa_distr

hh_income
less_than_35k        0.238510
35k_75k              0.313698
75k_100k             0.136306
greater_than_100k    0.311486
Name: pop, dtype: float64

In [10]:
cbg_distr = age.merge(income, on='GEOID', how='inner').dropna()
cbg_distr
# there are about 40 cbgs that are not overlapping between the income and the age data

,GEOID,18 to 24,25 to 44,45 to 66,67 and above,less_than_35k,35k_75k,75k_100k,greater_than_100k
0,040130101021,0.003542,0.319953,0.279811,0.396694,0.085106,0.127660,0.182033,0.605201
1,040130101022,0.006592,0.071852,0.555043,0.366513,0.295756,0.087533,0.049072,0.567639
2,040130101023,0.023019,0.000000,0.464151,0.512830,0.149068,0.103520,0.064182,0.683230
3,040130101031,0.045374,0.129004,0.556940,0.268683,0.226510,0.275168,0.082215,0.416107
4,040130101032,0.026382,0.236809,0.499372,0.237437,0.118758,0.140351,0.170040,0.570850
...,...,...,...,...,...,...,...,...,...
3028,040219413001,0.256350,0.389250,0.264028,0.090372,0.680303,0.157576,0.107576,0.054545
3029,040219414011,0.186841,0.299849,0.331994,0.181316,0.319481,0.270130,0.044156,0.366234
3030,040219414012,0.199516,0.408706,0.274486,0.117291,0.585915,0.273239,0.056338,0.084507
3031,040219414013,0.192424,0.283333,0.432323,0.091919,0.243065,0.459709,0.228534,0.068692


In [11]:
cbg_distr.to_csv('data/processed/cbg_acs_distr_38060.csv', index=False)